In [ ]:
from astropy.io import fits
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as u

# DR16Q
# fits_file = 'data/dr16q_prop_May01_2024.fits'
# hdul = fits.open(fits_file)
# data = hdul[1].data
# hdul.close()

# dr16q_coords = SkyCoord(ra=data['RA']*u.deg, dec=data['DEC']*u.deg)
# dr16q_objids = data['OBJID']


def generate_stone_lcs():
    # Load Stone et al. (2022) data
    fits_file = 'data/stone_TotalDat_v2.fits'
    hdul = fits.open(fits_file)
    data = hdul[1].data
    hdul.close()
    bands = ['g', 'r', 'i']
    fields = [
        'MAG', 'MAG_ERR', 'MJD',
        'log_SIGMA', 
        'log_TAU_REST',
    ]
    stone_lcs = {}

    for i in range(len(data)):
        dbid = data['DBID'][i]
        stone_lcs[dbid] = {
            'stone_DBID': dbid,
            'z': data['Z'][i],
            'stone_Z': data['Z'][i],
            'stone_RA': data['RA'][i],
            'stone_DEC': data['DEC'][i],
            'stone_LOG_M_BH': data['LOG_M_BH'][i],
            'stone_LOG_M_BH_ERR': data['LOG_M_BH_ERR'][i],
            'stone_LOG_LBOL': data['LOG_LBOL'][i],
            'stone_LOG_LBOL_ERR': data['LOG_LBOL_ERR'][i],        
            'mags': {},
            'magerrs': {},
            'times': {},
        }
        for band in bands:
            # Mask NaNs for MJD, MAG, and MAG_ERR fields
            mjd = data[f'MJD_{band}'][i]
            mag = data[f'MAG_{band}'][i]
            mag_err = data[f'MAG_ERR_{band}'][i]
            mask = ~np.isnan(mjd) & ~np.isnan(mag) & ~np.isnan(mag_err)

            stone_lcs[dbid]['mags'][band] = mag[mask]
            stone_lcs[dbid]['magerrs'][band] = mag_err[mask]
            stone_lcs[dbid]['times'][band] = mjd[mask]

            stone_lcs[dbid] |= {
                # f'MJD_{band}': mjd[mask],
                # f'MAG_{band}': mag[mask],
                # f'MAG_{band}_ERR': mag_err[mask],

                f'stone_log_SIGMA_{band}': data[f'log_SIGMA_{band}'][i],
                f'stone_log_SIGMA_{band}_ERR_L': data[f'log_SIGMA_{band}_ERR_L'][i],
                f'stone_log_SIGMA_{band}_ERR_U': data[f'log_SIGMA_{band}_ERR_U'][i],
                f'stone_log_TAU_REST_{band}': data[f'log_TAU_REST_{band}'][i],
                f'stone_log_TAU_REST_{band}_ERR_L': data[f'log_TAU_REST_{band}_ERR_L'][i],
                f'stone_log_TAU_REST_{band}_ERR_U': data[f'log_TAU_REST_{band}_ERR_U'][i],
                f'stone_log_SIGMA_{band}_ERR': (data[f'log_SIGMA_{band}_ERR_L'][i] + data[f'log_SIGMA_{band}_ERR_U'][i]) / 2,
                f'stone_log_TAU_REST_{band}_ERR': (data[f'log_TAU_REST_{band}_ERR_L'][i] + data[f'log_TAU_REST_{band}_ERR_U'][i]) / 2,
            }

    stone_coords = SkyCoord(ra=data['RA']*u.deg, dec=data['DEC']*u.deg)
    stone_ids = data['DBID']

    # S82 Catalog
    cat = pd.read_parquet("data/S82/Catalog.parquet").reset_index()
    cat_coords = SkyCoord(ra=cat['RA'].values*u.deg, dec=cat['DEC'].values*u.deg)
    cat_objids = cat['objectId'].values

    # Match lcs within 1 arcsec
    idx, d2d, _ = stone_coords.match_to_catalog_sky(cat_coords)
    match_mask = d2d < 1 * u.arcsec

    for i, matched in enumerate(match_mask):
        if matched:
            stone_lcs[stone_ids[i]]['object_id'] = cat_objids[idx[i]]
        else:
            stone_lcs[stone_ids[i]]['object_id'] = None
            print(f"Warning: No match found for DBID {stone_ids[i]}")
    return stone_lcs

stone_lcs = generate_stone_lcs()

In [73]:
cat.keys()

Index(['index', 'idx', 'ps1objID', 'uid', 'RA', 'DEC', 'PLATE', 'MJD',
       'FIBERID', 'Z_DR16Q', 'Z_SYS', 'IF_BOSS_SDSS', 'Nps1_g', 'Nps1_r',
       'Nps1_i', 'Nps1_z', 'Nps1_y', 'ps1raMean', 'ps1decMean', 'Nzuber_g',
       'Nzuber_r', 'objectId', 'Nsdss_u', 'Nsdss_g', 'Nsdss_r', 'Nsdss_i',
       'Nsdss_z', 'ebv', 'qg_LogL3000', 'qg_ebv', 'qg_fragal',
       'qg_LogL3000_err', 'qg_ebv_err', 'qg_fragal_err', 'sdss_g_qg',
       'sdss_r_qg', 'sdss_i_qg', 'sdss_z_qg', 'ps1_g_qg', 'ps1_r_qg',
       'ps1_i_qg', 'ps1_z_qg', 'ps1_y_qg'],
      dtype='object')

In [ ]:
# Convert stone_lcs to pandas DataFrame, ignoring 'mags', 'magerrs', 'times'
def stone_lcs_to_df(stone_lcs):
    records = []
    for obj in stone_lcs.values():
        rec = {k: v for k, v in obj.items() if k not in ['mags', 'magerrs', 'times']}
        records.append(rec)
    return pd.DataFrame(records)

stone_df = stone_lcs_to_df(stone_lcs)
# Resample stone_df to match the length of symmetric_err
resampled_df = stone_df.sample(n=len(stone_df), random_state=42).reset_index(drop=True)

# Add symmetric_err as a new column
resampled_df['symmetric_err'] = symmetric_err

# Write to CSV
resampled_df.to_csv('resampled_stone_lcs.csv', index=False)